# 🚢 Supply Chain Analytics — Full Project Notebook

---

| | |
|---|---|
| **Dataset** | `shipment.csv` — 728 rows, 12 columns |
| **Team Size** | 6 members |
| **Phases** | 8 (Business Questions → GenAI Optimization) |
| **Stack** | Python · Pandas · Matplotlib · Seaborn · Scikit-learn · SQLite |

---

## 📋 Table of Contents

| Phase | Title | Owner |
|-------|-------|-------|
| 1 | [Business Questions](#phase1) | Member 1 |
| 2 | [Data Audit & Cleaning](#phase2) | Member 2 |
| 3 | [Feature Engineering](#phase3) | Member 3 |
| 4 | [Exploratory Data Analysis](#phase4) | Member 4 |
| 5 | [Delay Probability Analysis](#phase5) | Member 5 |
| 6 | [Logistics Flow Visualization](#phase6) | Member 6 |
| 7 | [SQL Queries](#phase7) | Members 1 & 2 |
| 8 | [GenAI Optimization Suggestions](#phase8) | Members 3 & 4 |

---

> **How to use this notebook:**
> 1. Upload `shipment.csv` using the cell in Phase 2
> 2. Run cells top to bottom — each phase builds on the previous
> 3. Each phase is clearly marked with its owner
> 4. Do **not** skip Phase 2 — cleaning must happen before any analysis

---
## ⚙️ Setup — Install & Import All Libraries
**Run this first. Always.**

In [ ]:
# Install any libraries not in Colab by default
!pip install -q plotly kaleido

# ── Core ──────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualization ─────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Machine Learning ──────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# ── SQL ───────────────────────────────────────────
import sqlite3

# ── Display ───────────────────────────────────────
from IPython.display import display, HTML
from google.colab import files

# ── Plot style ────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

print('✅ All libraries loaded successfully.')

<a name='phase1'></a>

---
# 📌 Phase 1 — Define Business Questions
**Owner: Member 1**

Before touching any data, we define exactly what we're trying to answer. Every analysis in Phases 3–8 traces back to one of these questions.

In [ ]:
business_questions = {
    'Q1': 'Where are delays happening most often? Which origin, destination, or route is the highest risk?',
    'Q2': 'Which product category and shipment type (Import/Export) has the highest delay rate?',
    'Q3': 'Does longer customs clearance time lead to more shipment delays?',
    'Q4': 'Which trade routes are the most expensive (freight cost per unit value) and are they also the most delayed?',
}

print('=' * 70)
print('  SUPPLY CHAIN ANALYTICS — BUSINESS QUESTIONS')
print('=' * 70)
for k, v in business_questions.items():
    print(f'\n  {k}: {v}')
print()
print('  These questions drive every analytical decision in Phases 3–8.')
print('=' * 70)

<a name='phase2'></a>

---
# 🧹 Phase 2 — Data Audit & Cleaning
**Owner: Member 2**

This is the most critical phase. All downstream analysis depends on a clean dataset.

**Issues we will find and fix:**
| # | Column | Issue | Fix |
|---|--------|-------|-----|
| 1 | `delivery_status` | 24 NaN values | Rescued from corrupted column |
| 2 | `customs_clearance_time_days` | 24 rows contain text instead of numbers | Converted; unknown values → NaN |
| 3 | `D_Country` | 24 rows have numeric strings instead of country name | Set to 'Singapore' |
| 4 | `date` | Stored as plain string | Converted to datetime64 |
| 5 | All text columns | Inconsistent spacing and casing | Stripped and title-cased |

In [ ]:
# ── STEP 2.0 : Upload the dataset ─────────────────────────────────────────
print('Please upload shipment.csv when the file picker appears...')
uploaded = files.upload()
print('\n✅ File uploaded:', list(uploaded.keys()))

In [ ]:
# ── STEP 2.1 : Load & first look ──────────────────────────────────────────
df_raw = pd.read_csv('shipment.csv')

print(f'Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns\n')
display(df_raw.head(10))
print('\n── Column types & missing values ──')
audit = pd.DataFrame({
    'dtype':        df_raw.dtypes,
    'non_null':     df_raw.notnull().sum(),
    'missing':      df_raw.isnull().sum(),
    'missing_%':    (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'unique_vals':  df_raw.nunique()
})
display(audit)

In [ ]:
# ── STEP 2.2 : Identify the 24 corrupted rows ─────────────────────────────
#
# Root cause: 24 Industrial Equipment Export→Singapore shipments have a
# systematic 3-column data corruption:
#   - D_Country           → contains a number (e.g. "65000") instead of "Singapore"
#   - customs_clearance.. → contains "On-Time" or "Delayed" instead of a number
#   - delivery_status     → NaN (the value ended up in the wrong column)
#
# This is NOT random missingness — it's a patterned entry error.

corrupted_mask = df_raw['delivery_status'].isna()
print(f'Corrupted rows: {corrupted_mask.sum()}')
print(f'  All product_category : {df_raw.loc[corrupted_mask, "product_category"].unique()}')
print(f'  All type             : {df_raw.loc[corrupted_mask, "type"].unique()}')
print(f'  All destination      : {df_raw.loc[corrupted_mask, "destination"].unique()}')
print()
print('Sample of corrupted rows:')
display(df_raw[corrupted_mask][['shipment_id','D_Country','customs_clearance_time_days','delivery_status']].head(6))

In [ ]:
# ── STEP 2.3 : Apply all fixes ────────────────────────────────────────────
df = df_raw.copy()  # Never modify the raw dataframe — always work on a copy

# 2.3a — Flag corrupted rows BEFORE modifying anything
df['data_quality_flag'] = False
df.loc[corrupted_mask, 'data_quality_flag'] = True

# 2.3b — Rescue delivery_status from customs_clearance_time_days
df.loc[corrupted_mask, 'delivery_status'] = df.loc[corrupted_mask, 'customs_clearance_time_days']

# 2.3c — Fix D_Country for corrupted rows → 'Singapore'
df.loc[corrupted_mask, 'D_Country'] = 'Singapore'

# 2.3d — Clear out the bad customs values, then convert whole column to numeric
df.loc[corrupted_mask, 'customs_clearance_time_days'] = np.nan
df['customs_clearance_time_days'] = pd.to_numeric(df['customs_clearance_time_days'], errors='coerce')

# 2.3e — Convert date to datetime
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='coerce')

# 2.3f — Standardize all text columns: strip whitespace, title case
text_cols = ['type', 'product_category', 'origin', 'O_Country',
             'destination', 'D_Country', 'delivery_status']
for col in text_cols:
    df[col] = df[col].str.strip().str.title()

# 2.3g — Fix 'Usa' → 'USA' (title() side effect)
df['O_Country'] = df['O_Country'].replace({'Usa': 'USA'})
df['D_Country']  = df['D_Country'].replace({'Usa': 'USA'})

print('✅ All fixes applied.')
print(f'   Rows: {len(df)} | Columns: {len(df.columns)}')
print(f'   data_quality_flag=True: {df["data_quality_flag"].sum()} rows')

In [ ]:
# ── STEP 2.4 : Post-cleaning validation ───────────────────────────────────
print('── Missing values after cleaning ──')
missing_after = df.isnull().sum()
print(missing_after[missing_after > 0].to_string() or '  None!')
print()

print('── delivery_status distribution ──')
print(df['delivery_status'].value_counts(dropna=False).to_string())
print()

print('── D_Country distribution ──')
print(df['D_Country'].value_counts().to_string())
print()

print('── customs_clearance_time_days stats ──')
print(df['customs_clearance_time_days'].describe().round(2).to_string())
print()

print('── Date range ──')
print(f'  From: {df["date"].min().date()}  →  To: {df["date"].max().date()}')
print()

print('── Duplicate rows ──')
print(f'  {df.duplicated().sum()} duplicates found')

In [ ]:
# ── STEP 2.5 : Save cleaned dataset ───────────────────────────────────────
df.to_csv('shipment_cleaned.csv', index=False)
files.download('shipment_cleaned.csv')
print('✅ Cleaned dataset saved as shipment_cleaned.csv')
print('   Hand this file to Member 3 for feature engineering.')

<a name='phase3'></a>

---
# 🔧 Phase 3 — Feature Engineering
**Owner: Member 3**

We create new columns that make analysis easier and more powerful. Raw columns like `origin` and `destination` become `route`. The binary `delivery_status` becomes `is_delayed` (0/1) which is required for modelling.

In [ ]:
# ── STEP 3.1 : Time-based features ────────────────────────────────────────
df['year']    = df['date'].dt.year
df['month']   = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df['month_name'] = df['date'].dt.strftime('%b')  # Jan, Feb, ...

print('✅ Time features created: year, month, quarter, month_name')

# ── STEP 3.2 : Route and lane ─────────────────────────────────────────────
df['route'] = df['origin'] + ' → ' + df['destination']
df['lane']  = df['O_Country'] + ' → ' + df['D_Country']

print('✅ Route features created: route (city-level), lane (country-level)')

# ── STEP 3.3 : Binary delay flag ──────────────────────────────────────────
# 1 = Delayed, 0 = On-Time — required for probability analysis and ML
df['is_delayed'] = (df['delivery_status'] == 'Delayed').astype(int)

print(f'✅ is_delayed created: {df["is_delayed"].sum()} delayed, {(df["is_delayed"]==0).sum()} on-time')

# ── STEP 3.4 : High-value shipment flag ───────────────────────────────────
# Threshold: top 25% of shipment values (75th percentile)
value_threshold = df['value'].quantile(0.75)
df['high_value_shipment'] = (df['value'] >= value_threshold).astype(int)

print(f'✅ high_value_shipment created: threshold = ₹{value_threshold:,.0f}')

# ── STEP 3.5 : Freight efficiency ─────────────────────────────────────────
# freight_per_value = what fraction of shipment value goes to freight
# Lower is more efficient. Helps identify expensive routes.
df['freight_per_value'] = (df['freight_cost'] / df['value']).round(4)

print('✅ freight_per_value created (freight cost / shipment value)')

# ── STEP 3.6 : Customs clearance bucket ───────────────────────────────────
# Converts continuous clearance time into Low / Medium / High category
df['clearance_bucket'] = pd.cut(
    df['customs_clearance_time_days'],
    bins=[0, 2.5, 4.0, 10],
    labels=['Low (≤2.5 days)', 'Medium (2.5–4 days)', 'High (>4 days)']
)

print('✅ clearance_bucket created: Low / Medium / High')
print(df['clearance_bucket'].value_counts().to_string())

In [ ]:
# ── STEP 3.7 : Preview engineered dataset ─────────────────────────────────
new_cols = ['route', 'lane', 'is_delayed', 'high_value_shipment',
            'freight_per_value', 'clearance_bucket', 'year', 'month', 'quarter']
print(f'New columns added: {len(new_cols)}')
print(f'Total columns now: {len(df.columns)}')
print()
display(df[['shipment_id', 'route', 'is_delayed', 'high_value_shipment',
            'freight_per_value', 'clearance_bucket']].head(10))

<a name='phase4'></a>

---
# 📊 Phase 4 — Exploratory Data Analysis (EDA)
**Owner: Member 4**

We explore the dataset in three layers:
1. **Univariate** — distribution of individual columns
2. **Bivariate** — relationships between pairs of columns
3. **Time-based** — patterns over months and quarters

In [ ]:
# ── 4.1 : Univariate — Shipment value distribution ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['value'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Shipment Value Distribution')
axes[0].set_xlabel('Value (USD)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

axes[1].hist(df['freight_cost'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Freight Cost Distribution')
axes[1].set_xlabel('Freight Cost (USD)')
axes[1].set_ylabel('Count')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('plot_01_univariate_distributions.png', bbox_inches='tight')
plt.show()
print('📌 Observation: Shipment values are right-skewed. Most shipments cluster between $50K–$200K.')

In [ ]:
# ── 4.2 : Univariate — Delivery status & customs clearance ────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Pie chart for delivery status
status_counts = df['delivery_status'].value_counts()
axes[0].pie(status_counts, labels=status_counts.index,
            autopct='%1.1f%%', colors=['#4CAF50', '#F44336'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Delivery Status (Overall)')

# Customs clearance time histogram
axes[1].hist(df['customs_clearance_time_days'].dropna(), bins=20,
             color='mediumpurple', edgecolor='white')
axes[1].set_title('Customs Clearance Time Distribution')
axes[1].set_xlabel('Days')
axes[1].set_ylabel('Count')
axes[1].axvline(df['customs_clearance_time_days'].mean(), color='red',
                linestyle='--', linewidth=1.5, label=f'Mean: {df["customs_clearance_time_days"].mean():.1f} days')
axes[1].legend()

plt.tight_layout()
plt.savefig('plot_02_status_customs.png', bbox_inches='tight')
plt.show()
print(f'📌 Observation: {status_counts["Delayed"]/status_counts.sum()*100:.1f}% of shipments are delayed. Mean customs clearance = {df["customs_clearance_time_days"].mean():.1f} days.')

In [ ]:
# ── 4.3 : Bivariate — Delay rate by product category & shipment type ──────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# By product category
cat_delay = df.groupby('product_category')['is_delayed'].mean().sort_values(ascending=False) * 100
bars1 = axes[0].bar(cat_delay.index, cat_delay.values,
                    color=['#E74C3C' if v > 15 else '#3498DB' for v in cat_delay.values],
                    edgecolor='white')
axes[0].set_title('Delay Rate by Product Category')
axes[0].set_ylabel('Delay Rate (%)')
axes[0].tick_params(axis='x', rotation=15)
for bar, val in zip(bars1, cat_delay.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# By shipment type
type_delay = df.groupby('type')['is_delayed'].mean().sort_values(ascending=False) * 100
bars2 = axes[1].bar(type_delay.index, type_delay.values,
                    color=['#E74C3C', '#3498DB'], edgecolor='white')
axes[1].set_title('Delay Rate by Shipment Type')
axes[1].set_ylabel('Delay Rate (%)')
for bar, val in zip(bars2, type_delay.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('plot_03_delay_by_category_type.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.4 : Bivariate — Customs clearance time vs delay status ──────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Box plot
clean_df = df.dropna(subset=['customs_clearance_time_days'])
sns.boxplot(data=clean_df, x='delivery_status', y='customs_clearance_time_days',
            palette={'On-Time': '#4CAF50', 'Delayed': '#F44336'}, ax=axes[0])
axes[0].set_title('Customs Clearance Time vs Delivery Status')
axes[0].set_xlabel('Delivery Status')
axes[0].set_ylabel('Customs Clearance (days)')

# Delay rate by clearance bucket
bucket_delay = clean_df.groupby('clearance_bucket', observed=True)['is_delayed'].mean() * 100
bars = axes[1].bar(bucket_delay.index.astype(str), bucket_delay.values,
                   color=['#4CAF50', '#FFA726', '#F44336'], edgecolor='white')
axes[1].set_title('Delay Rate by Customs Clearance Bucket')
axes[1].set_ylabel('Delay Rate (%)')
axes[1].tick_params(axis='x', rotation=10)
for bar, val in zip(bars, bucket_delay.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('plot_04_customs_vs_delay.png', bbox_inches='tight')
plt.show()

delayed_mean = clean_df[clean_df['is_delayed']==1]['customs_clearance_time_days'].mean()
ontime_mean  = clean_df[clean_df['is_delayed']==0]['customs_clearance_time_days'].mean()
print(f'📌 Delayed shipments avg clearance: {delayed_mean:.2f} days')
print(f'📌 On-time shipments avg clearance: {ontime_mean:.2f} days')

In [ ]:
# ── 4.5 : Bivariate — Top delayed origin countries ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top origin countries by delay rate
origin_stats = df.groupby('O_Country').agg(
    total=('is_delayed', 'count'),
    delayed=('is_delayed', 'sum')
).reset_index()
origin_stats['delay_rate'] = (origin_stats['delayed'] / origin_stats['total'] * 100).round(1)
origin_stats = origin_stats[origin_stats['total'] >= 20].sort_values('delay_rate', ascending=True)

axes[0].barh(origin_stats['O_Country'], origin_stats['delay_rate'],
             color=['#E74C3C' if v > 20 else '#3498DB' for v in origin_stats['delay_rate']],
             edgecolor='white')
axes[0].set_title('Delay Rate by Origin Country')
axes[0].set_xlabel('Delay Rate (%)')
axes[0].axvline(df['is_delayed'].mean()*100, color='gray', linestyle='--', linewidth=1, label='Overall avg')
axes[0].legend()

# Top destination countries by delay rate
dest_stats = df.groupby('D_Country').agg(
    total=('is_delayed', 'count'),
    delayed=('is_delayed', 'sum')
).reset_index()
dest_stats['delay_rate'] = (dest_stats['delayed'] / dest_stats['total'] * 100).round(1)
dest_stats = dest_stats[dest_stats['total'] >= 10].sort_values('delay_rate', ascending=True)

axes[1].barh(dest_stats['D_Country'], dest_stats['delay_rate'],
             color=['#E74C3C' if v > 20 else '#3498DB' for v in dest_stats['delay_rate']],
             edgecolor='white')
axes[1].set_title('Delay Rate by Destination Country')
axes[1].set_xlabel('Delay Rate (%)')
axes[1].axvline(df['is_delayed'].mean()*100, color='gray', linestyle='--', linewidth=1, label='Overall avg')
axes[1].legend()

plt.tight_layout()
plt.savefig('plot_05_delay_by_country.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.6 : Time-based — Monthly delay trend ────────────────────────────────
monthly = df.groupby(['year', 'month', 'month_name'])['is_delayed'].agg(['mean', 'sum', 'count']).reset_index()
monthly.columns = ['year', 'month', 'month_name', 'delay_rate', 'delayed_count', 'total']
monthly['delay_rate'] = monthly['delay_rate'] * 100
monthly = monthly.sort_values(['year', 'month'])
monthly['period'] = monthly['year'].astype(str) + '-' + monthly['month_name']

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(monthly['period'], monthly['delay_rate'],
        color='#E74C3C', linewidth=2, marker='o', markersize=5)
ax.fill_between(monthly['period'], monthly['delay_rate'], alpha=0.15, color='#E74C3C')
ax.axhline(df['is_delayed'].mean()*100, color='gray', linestyle='--', linewidth=1,
           label=f'Overall avg ({df["is_delayed"].mean()*100:.1f}%)')
ax.set_title('Monthly Shipment Delay Rate (2024–2025)')
ax.set_ylabel('Delay Rate (%)')
ax.set_xlabel('Month')
ax.tick_params(axis='x', rotation=60)
ax.legend()
plt.tight_layout()
plt.savefig('plot_06_monthly_delay_trend.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 4.7 : Freight cost vs value scatter ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors = {'Delayed': '#E74C3C', 'On-Time': '#3498DB'}
for status, grp in df.groupby('delivery_status'):
    ax.scatter(grp['value'], grp['freight_cost'], alpha=0.4, s=18,
               color=colors[status], label=status)
ax.set_title('Freight Cost vs Shipment Value')
ax.set_xlabel('Shipment Value (USD)')
ax.set_ylabel('Freight Cost (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend(title='Delivery Status')
plt.tight_layout()
plt.savefig('plot_07_freight_vs_value.png', bbox_inches='tight')
plt.show()
print('📌 Observation: Freight cost is almost exactly 5% of shipment value for most rows — very consistent pricing.')

In [ ]:
# ── 4.8 : EDA Summary table ───────────────────────────────────────────────
print('=' * 60)
print('  EDA SUMMARY')
print('=' * 60)
print(f'  Total shipments       : {len(df)}')
print(f'  Date range            : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'  Overall delay rate    : {df["is_delayed"].mean()*100:.1f}%')
print(f'  Avg shipment value    : ${df["value"].mean():,.0f}')
print(f'  Avg freight cost      : ${df["freight_cost"].mean():,.0f}')
print(f'  Avg customs clearance : {df["customs_clearance_time_days"].mean():.2f} days')
print(f'  Most delayed category : {df.groupby("product_category")["is_delayed"].mean().idxmax()}')
print(f'  Most delayed origin   : {df.groupby("O_Country")["is_delayed"].mean().idxmax()}')
print('=' * 60)

<a name='phase5'></a>

---
# 🎯 Phase 5 — Delay Probability Analysis
**Owner: Member 5**

We move from *describing* delays to *quantifying* and *predicting* them. Two layers:
1. **Conditional probability tables** — delay rate broken down by each variable
2. **Predictive model** — Logistic Regression + Decision Tree to predict delay probability for a new shipment

In [ ]:
# ── 5.1 : Conditional probability tables ──────────────────────────────────
print('── DELAY PROBABILITY TABLES ──\n')

def delay_prob_table(groupby_col, label):
    tbl = df.groupby(groupby_col).agg(
        total        = ('is_delayed', 'count'),
        delayed      = ('is_delayed', 'sum'),
        delay_prob   = ('is_delayed', 'mean')
    ).reset_index()
    tbl['delay_prob_%'] = (tbl['delay_prob'] * 100).round(1)
    tbl = tbl.sort_values('delay_prob_%', ascending=False)
    print(f'  {label}')
    print(tbl[[groupby_col, 'total', 'delayed', 'delay_prob_%']].to_string(index=False))
    print()
    return tbl

t1 = delay_prob_table('product_category', 'By Product Category')
t2 = delay_prob_table('type',             'By Shipment Type')
t3 = delay_prob_table('O_Country',        'By Origin Country')
t4 = delay_prob_table('D_Country',        'By Destination Country')

In [ ]:
# ── 5.2 : Delay probability by clearance bucket ───────────────────────────
clean_df = df.dropna(subset=['customs_clearance_time_days', 'clearance_bucket'])
t5 = delay_prob_table.__wrapped__ if hasattr(delay_prob_table, '__wrapped__') else None

bucket_tbl = clean_df.groupby('clearance_bucket', observed=True).agg(
    total      = ('is_delayed', 'count'),
    delayed    = ('is_delayed', 'sum'),
    delay_prob = ('is_delayed', 'mean')
).reset_index()
bucket_tbl['delay_prob_%'] = (bucket_tbl['delay_prob'] * 100).round(1)
print('  By Customs Clearance Bucket')
print(bucket_tbl[['clearance_bucket', 'total', 'delayed', 'delay_prob_%']].to_string(index=False))

In [ ]:
# ── 5.3 : Top 10 highest-risk routes ──────────────────────────────────────
route_stats = df.groupby('route').agg(
    total        = ('is_delayed', 'count'),
    delayed      = ('is_delayed', 'sum'),
    delay_prob   = ('is_delayed', 'mean'),
    avg_freight  = ('freight_cost', 'mean')
).reset_index()
route_stats['delay_prob_%'] = (route_stats['delay_prob'] * 100).round(1)
top_routes = route_stats[route_stats['total'] >= 3].sort_values('delay_prob_%', ascending=False).head(10)

print('  Top 10 Highest-Risk Routes (min 3 shipments)')
display(top_routes[['route', 'total', 'delayed', 'delay_prob_%', 'avg_freight']].reset_index(drop=True))

In [ ]:
# ── 5.4 : Predictive model ────────────────────────────────────────────────
# Prepare features — encode categorical columns
model_df = df.dropna(subset=['customs_clearance_time_days']).copy()

le = LabelEncoder()
features = ['type', 'product_category', 'O_Country', 'D_Country',
            'customs_clearance_time_days', 'freight_per_value', 'high_value_shipment']

X = model_df[features].copy()
for col in ['type', 'product_category', 'O_Country', 'D_Country']:
    X[col] = le.fit_transform(X[col])

y = model_df['is_delayed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Logistic Regression
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Decision Tree
dt = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print('── Logistic Regression Results ──')
print(classification_report(y_test, y_pred_lr, target_names=['On-Time', 'Delayed']))

print('── Decision Tree Results ──')
print(classification_report(y_test, y_pred_dt, target_names=['On-Time', 'Delayed']))

In [ ]:
# ── 5.5 : Feature importance (Decision Tree) ──────────────────────────────
feat_importance = pd.DataFrame({
    'feature':    features,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(feat_importance['feature'], feat_importance['importance'],
        color='steelblue', edgecolor='white')
ax.set_title('Feature Importance — Decision Tree')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('plot_08_feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.6 : Confusion matrix ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, y_pred, title in [
    (axes[0], y_pred_lr, 'Logistic Regression'),
    (axes[1], y_pred_dt, 'Decision Tree')
]:
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['On-Time', 'Delayed'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)

plt.tight_layout()
plt.savefig('plot_09_confusion_matrices.png', bbox_inches='tight')
plt.show()

<a name='phase6'></a>

---
# 🗺️ Phase 6 — Logistics Flow Visualization
**Owner: Member 6**

The storytelling phase — interactive visuals that show where shipments flow, where delays concentrate, and which routes are most costly.

In [ ]:
# ── 6.1 : Sankey diagram — country-level trade flow ───────────────────────
lane_counts = df.groupby(['O_Country', 'D_Country']).size().reset_index(name='count')
lane_counts = lane_counts[lane_counts['count'] >= 5]

all_nodes = pd.unique(lane_counts[['O_Country', 'D_Country']].values.ravel()).tolist()
node_idx  = {n: i for i, n in enumerate(all_nodes)}

fig = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(
        pad=20, thickness=18,
        label=all_nodes,
        color='#3498DB'
    ),
    link=dict(
        source=[node_idx[r] for r in lane_counts['O_Country']],
        target=[node_idx[r] for r in lane_counts['D_Country']],
        value=lane_counts['count'],
        color='rgba(52, 152, 219, 0.3)'
    )
))
fig.update_layout(
    title='Shipment Flow — Origin → Destination Countries',
    font_size=12, height=500
)
fig.show()
fig.write_html('plot_10_sankey.html')
print('✅ Sankey saved as plot_10_sankey.html')

In [ ]:
# ── 6.2 : Heatmap — Delay rate by Origin × Destination country ────────────
pivot = df.groupby(['O_Country', 'D_Country'])['is_delayed'].mean().unstack(fill_value=0) * 100

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Delay Rate (%)'},
            ax=ax)
ax.set_title('Delay Rate Heatmap — Origin × Destination Country (%)', pad=14)
ax.set_xlabel('Destination Country')
ax.set_ylabel('Origin Country')
ax.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.savefig('plot_11_delay_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.3 : Interactive bar — top delayed routes (Plotly) ───────────────────
top15 = route_stats.sort_values('delay_prob_%', ascending=False).head(15)

fig = px.bar(
    top15, x='delay_prob_%', y='route', orientation='h',
    color='delay_prob_%', color_continuous_scale='Reds',
    labels={'delay_prob_%': 'Delay Rate (%)', 'route': 'Route'},
    title='Top 15 Highest-Risk Trade Routes',
    text='delay_prob_%'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=520, yaxis={'categoryorder': 'total ascending'},
                  coloraxis_showscale=False)
fig.show()
fig.write_html('plot_12_top_routes.html')

In [ ]:
# ── 6.4 : Interactive world map — delay rate by destination ───────────────
dest_map = df.groupby('D_Country').agg(
    delay_rate=('is_delayed', 'mean'),
    total=('is_delayed', 'count')
).reset_index()
dest_map['delay_rate_%'] = (dest_map['delay_rate'] * 100).round(1)

fig = px.choropleth(
    dest_map,
    locations='D_Country',
    locationmode='country names',
    color='delay_rate_%',
    hover_name='D_Country',
    hover_data={'total': True, 'delay_rate_%': True},
    color_continuous_scale='OrRd',
    title='Delay Rate by Destination Country',
    labels={'delay_rate_%': 'Delay Rate (%)'}
)
fig.update_layout(height=460)
fig.show()
fig.write_html('plot_13_world_map.html')
print('✅ Interactive world map saved.')

In [ ]:
# ── 6.5 : Stacked bar — delayed vs on-time by category & type ─────────────
cross = pd.crosstab(df['product_category'], df['delivery_status'])

fig, ax = plt.subplots(figsize=(9, 4))
cross.plot(kind='bar', stacked=True, ax=ax,
           color=['#E74C3C', '#4CAF50'], edgecolor='white')
ax.set_title('Shipment Count by Product Category & Delivery Status')
ax.set_xlabel('')
ax.set_ylabel('Number of Shipments')
ax.tick_params(axis='x', rotation=15)
ax.legend(title='Status')
plt.tight_layout()
plt.savefig('plot_14_stacked_category.png', bbox_inches='tight')
plt.show()

<a name='phase7'></a>

---
# 🗄️ Phase 7 — SQL Queries
**Owners: Members 1 & 2**

We load the cleaned data into an in-memory SQLite database and run business-relevant queries. This demonstrates how the data would be queried in a real production database.

In [ ]:
# ── 7.0 : Create SQLite database and load table ───────────────────────────
conn = sqlite3.connect(':memory:')

# SQLite doesn't support pandas datetime — convert back to string
df_sql = df.copy()
df_sql['date'] = df_sql['date'].dt.strftime('%Y-%m-%d')
df_sql['clearance_bucket'] = df_sql['clearance_bucket'].astype(str)
df_sql.to_sql('shipments', conn, if_exists='replace', index=False)

print(f'✅ SQLite table "shipments" created with {len(df_sql)} rows.')
print('   Schema:')
schema = pd.read_sql('PRAGMA table_info(shipments)', conn)
print(schema[['name','type']].to_string(index=False))

In [ ]:
# ── Q1 : Which routes have the most delayed shipments? ────────────────────
q1 = '''
SELECT
    origin || ' → ' || destination   AS route,
    COUNT(*)                          AS total_shipments,
    SUM(is_delayed)                   AS delayed_count,
    ROUND(AVG(is_delayed) * 100, 1)   AS delay_rate_pct
FROM shipments
GROUP BY route
HAVING total_shipments >= 3
ORDER BY delay_rate_pct DESC
LIMIT 10;
'''
print('Q1: Top 10 routes by delay rate')
display(pd.read_sql(q1, conn))

In [ ]:
# ── Q2 : Average freight cost by product category ─────────────────────────
q2 = '''
SELECT
    product_category,
    COUNT(*)                              AS shipments,
    ROUND(AVG(freight_cost), 0)           AS avg_freight_usd,
    ROUND(AVG(value), 0)                  AS avg_value_usd,
    ROUND(AVG(freight_cost / value), 4)   AS avg_freight_ratio
FROM shipments
GROUP BY product_category
ORDER BY avg_freight_usd DESC;
'''
print('Q2: Average freight cost by product category')
display(pd.read_sql(q2, conn))

In [ ]:
# ── Q3 : Which origin countries have the highest delay rate? ──────────────
q3 = '''
SELECT
    O_Country,
    COUNT(*)                            AS total_shipments,
    SUM(is_delayed)                     AS delayed,
    ROUND(AVG(is_delayed) * 100, 1)     AS delay_rate_pct,
    ROUND(AVG(customs_clearance_time_days), 2) AS avg_clearance_days
FROM shipments
WHERE customs_clearance_time_days IS NOT NULL
GROUP BY O_Country
ORDER BY delay_rate_pct DESC;
'''
print('Q3: Delay rate and clearance time by origin country')
display(pd.read_sql(q3, conn))

In [ ]:
# ── Q4 : Which destination country receives the most shipments? ────────────
q4 = '''
SELECT
    D_Country,
    COUNT(*)                         AS total_shipments,
    SUM(CASE WHEN type='Import' THEN 1 ELSE 0 END) AS imports,
    SUM(CASE WHEN type='Export' THEN 1 ELSE 0 END) AS exports,
    ROUND(AVG(is_delayed) * 100, 1)  AS delay_rate_pct,
    ROUND(SUM(value) / 1000000.0, 2) AS total_value_million_usd
FROM shipments
GROUP BY D_Country
ORDER BY total_shipments DESC;
'''
print('Q4: Shipment volume and delay rate by destination country')
display(pd.read_sql(q4, conn))

In [ ]:
# ── Q5 : Monthly delayed shipment trend ───────────────────────────────────
q5 = '''
SELECT
    SUBSTR(date, 1, 7)              AS year_month,
    COUNT(*)                         AS total,
    SUM(is_delayed)                  AS delayed,
    ROUND(AVG(is_delayed) * 100, 1)  AS delay_rate_pct
FROM shipments
GROUP BY year_month
ORDER BY year_month;
'''
print('Q5: Monthly delayed shipment trend')
display(pd.read_sql(q5, conn))

In [ ]:
# ── Q6 : High-value shipments that are delayed most often ─────────────────
q6 = '''
SELECT
    product_category,
    O_Country,
    D_Country,
    COUNT(*)                          AS high_value_shipments,
    SUM(is_delayed)                   AS delayed_count,
    ROUND(AVG(is_delayed) * 100, 1)   AS delay_rate_pct,
    ROUND(AVG(value), 0)              AS avg_value_usd
FROM shipments
WHERE high_value_shipment = 1
GROUP BY product_category, O_Country, D_Country
HAVING high_value_shipments >= 3
ORDER BY delay_rate_pct DESC
LIMIT 10;
'''
print('Q6: High-value shipments with highest delay rates')
display(pd.read_sql(q6, conn))

<a name='phase8'></a>

---
# 🤖 Phase 8 — GenAI Optimization Suggestions
**Owners: Members 3 & 4**

We synthesize findings from all previous phases and use them to generate actionable, data-grounded recommendations. Two modes:
1. **Rule-based summaries** — computed directly from our analysis
2. **LLM-powered suggestions** — structured prompts fed to an AI model (Claude API or OpenAI)

In [ ]:
# ── 8.1 : Build a structured insight summary ──────────────────────────────
# This is the clean input we'll feed to the LLM. Never send raw data — send summaries.

overall_delay_rate = round(df['is_delayed'].mean() * 100, 1)
top_delayed_cat    = df.groupby('product_category')['is_delayed'].mean().idxmax()
top_delayed_cat_r  = round(df.groupby('product_category')['is_delayed'].mean().max() * 100, 1)
top_origin         = df.groupby('O_Country')['is_delayed'].mean().idxmax()
top_origin_r       = round(df.groupby('O_Country')['is_delayed'].mean().max() * 100, 1)
top_dest           = df.groupby('D_Country')['is_delayed'].mean().idxmax()
top_dest_r         = round(df.groupby('D_Country')['is_delayed'].mean().max() * 100, 1)
top_route          = route_stats.sort_values('delay_prob_%', ascending=False).iloc[0]
avg_clearance      = round(df['customs_clearance_time_days'].mean(), 2)
del_clearance      = round(df[df['is_delayed']==1]['customs_clearance_time_days'].mean(), 2)

insight_summary = f"""
SUPPLY CHAIN ANALYTICS — KEY FINDINGS SUMMARY
Dataset: 728 shipments, 2024–2025, India-centric trade

1. Overall delay rate: {overall_delay_rate}% of all shipments are delayed.

2. Product category risk:
   - Highest delay rate: {top_delayed_cat} at {top_delayed_cat_r}%

3. Geographic risk:
   - Highest delay origin: {top_origin} ({top_origin_r}% delay rate)
   - Highest delay destination: {top_dest} ({top_dest_r}% delay rate)

4. Riskiest single route: {top_route['route']} — {top_route['delay_prob_%']}% delay rate

5. Customs clearance impact:
   - Average clearance time overall: {avg_clearance} days
   - Average clearance time for DELAYED shipments: {del_clearance} days
   - High-clearance shipments (>4 days) have significantly higher delay rates.

6. Freight cost: Almost uniformly 5% of shipment value — no significant
   cost variation by route or category.
"""

print(insight_summary)

In [ ]:
# ── 8.2 : Rule-based optimization recommendations ─────────────────────────
# Generated from our own analysis — no API key needed

recommendations = [
    {
        'priority': '🔴 HIGH',
        'finding':  f'{top_delayed_cat} has the highest delay rate ({top_delayed_cat_r}%)',
        'action':   f'Initiate pre-clearance documentation for all {top_delayed_cat} shipments '
                    f'at least 48 hours before dispatch. Assign a dedicated customs agent.'
    },
    {
        'priority': '🔴 HIGH',
        'finding':  f'Shipments with >4 days customs clearance are disproportionately delayed',
        'action':   'Implement automated document review for high-clearance-risk categories. '
                    'Flag any shipment predicted to exceed 4 days clearance for expedited processing.'
    },
    {
        'priority': '🟡 MEDIUM',
        'finding':  f'Route "{top_route["route"]}" has {top_route["delay_prob_%"]}% delay rate',
        'action':   'Evaluate alternate carriers or logistics providers for this lane. '
                    'Consider route rescheduling to avoid peak customs periods.'
    },
    {
        'priority': '🟡 MEDIUM',
        'finding':  f'{top_origin} origin shipments show elevated delay risk ({top_origin_r}%)',
        'action':   f'Review vendor lead times and packaging compliance for {top_origin} suppliers. '
                    'Build additional buffer days into SLAs for this origin.'
    },
    {
        'priority': '🟢 LOW',
        'finding':  'Freight cost is uniformly ~5% of value with no variation',
        'action':   'Benchmark against market rates. Flat-rate pricing suggests no dynamic routing '
                    'optimization is being applied — opportunity to negotiate volume-tiered rates.'
    },
]

print('=' * 70)
print('  SUPPLY CHAIN OPTIMIZATION RECOMMENDATIONS')
print('=' * 70)
for i, rec in enumerate(recommendations, 1):
    print(f'\n[{rec["priority"]}] Recommendation {i}')
    print(f'  Finding : {rec["finding"]}')
    print(f'  Action  : {rec["action"]}')
print()

In [ ]:
# ── 8.3 : LLM-powered suggestions via Claude API ──────────────────────────
# Optional — requires an Anthropic API key
# Get a free API key at https://console.anthropic.com

import os
import json

ANTHROPIC_API_KEY = ''  # ← Paste your API key here, or set as env variable

if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')

if ANTHROPIC_API_KEY:
    !pip install -q anthropic
    import anthropic

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    prompt = f"""
You are a supply chain consultant. Based on the following analytics findings,
provide 5 specific, actionable recommendations to reduce shipment delays
and improve logistics efficiency. Each recommendation should name the
specific metric or finding it addresses.

FINDINGS:
{insight_summary}

Format each recommendation as:
RECOMMENDATION [N]: [Title]
Problem: [what the data shows]
Action: [specific step to take]
Expected Impact: [measurable outcome]
"""

    message = client.messages.create(
        model='claude-sonnet-4-20250514',
        max_tokens=1000,
        messages=[{'role': 'user', 'content': prompt}]
    )

    print('── Claude API Response ──')
    print(message.content[0].text)
else:
    print('ℹ️  No API key provided. Paste your Anthropic API key into ANTHROPIC_API_KEY above.')
    print('   Rule-based recommendations from Step 8.2 are still complete and valid for your project.')

---
# ✅ Project Complete

## What was delivered

| Phase | Output |
|-------|--------|
| 1 | 4 business questions guiding the entire analysis |
| 2 | Cleaned dataset (728 rows, 0 missing delivery_status) |
| 3 | 9 new features including `is_delayed`, `route`, `clearance_bucket` |
| 4 | 7 EDA charts covering distributions, bivariate, and time trends |
| 5 | Delay probability tables + Logistic Regression + Decision Tree |
| 6 | Sankey diagram, heatmap, world map, stacked bar charts |
| 7 | 6 SQL queries on an in-memory SQLite database |
| 8 | Rule-based + optional LLM-powered optimization recommendations |

## Files generated
- `shipment_cleaned.csv` — cleaned dataset for handoff
- `plot_01` to `plot_14` — all charts (PNG + interactive HTML)
- `plot_10_sankey.html` — interactive Sankey diagram
- `plot_12_top_routes.html` — interactive delay route chart
- `plot_13_world_map.html` — interactive choropleth map

---
*Supply Chain Analytics Project — Team of 6*